In [8]:
import pandas as pd
import numpy as np
import os
import unicodedata

# --- 1. CONFIGURATION ---
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"
os.makedirs(PROCESSED_PATH, exist_ok=True)

files = {
    'gdp_primary': 'Valeur ajoutée du secteur primaire - Tableau.xlsx',
    'gdp_secondary': 'Valeur ajoutée du secteur secondaire - Tableau.xlsx',
    'gdp_tertiary': 'Valeur ajoutée du secteur tertiaire - Tableau.xlsx',
    'gdp_total': 'PIB en volume - Tableau.xlsx',
    'emp_branches': 'Emploi par branche au niveau national - Annuel.xlsx'
}

# --- 2. UTILS ---
def normalize_text(text):
    text = str(text).lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    return text

def find_sector_columns(columns):
    col_map = {}

    for col in columns:
        col_norm = normalize_text(col)

        if any(k in col_norm for k in ['primaire', 'agriculture']):
            col_map['Primary'] = col

        elif any(k in col_norm for k in ['secondaire', 'industrie']):
            col_map['Secondary'] = col

        elif any(k in col_norm for k in ['tertiaire', 'services']):
            col_map['Tertiary'] = col

    if len(col_map) != 3:
        raise ValueError(f"❌ Sector columns not detected correctly: {col_map}")

    return col_map


def find_growth_column(df):
    for col in df.columns:
        col_norm = normalize_text(col)
        if any(k in col_norm for k in ['taux', 'croissance', 'variation']):
            return col
    raise ValueError(f"❌ Growth column not found: {df.columns}")


# --- 3. CORE FUNCTIONS ---
def build_index(growth_series, start_val=100.0):
    sorted_rates = growth_series.iloc[::-1].fillna(0)
    levels = [start_val]

    for rate in sorted_rates:
        levels.append(levels[-1] * (1 + rate / 100))

    return levels[1:][::-1]


def robust_clean_hcp(df):
    header_idx = None

    for i, row in enumerate(df.values):
        if any(str(val).strip().startswith('Année') for val in row):
            header_idx = i
            break

    if header_idx is not None:
        df.columns = df.iloc[header_idx].astype(str).str.strip()
        df = df.iloc[header_idx + 1:].copy()

    # Year column
    y_col = next((c for c in df.columns if str(c).startswith('Année')), None)

    if y_col:
        df = df.rename(columns={y_col: 'Year'})
        df['Year'] = (
            df['Year']
            .astype(str)
            .str.replace('*', '', regex=False)
            .str.strip()
        )
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

    # Drop junk columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed', case=False, na=False)]

    # Clean numeric values
    for col in df.columns:
        if col != 'Year':
            df[col] = (
                df[col]
                .astype(str)
                .str.replace('%', '', regex=False)
                .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna(subset=['Year']).reset_index(drop=True)


# --- 4. PIPELINE ---
print("🚀 Launching Pipeline...")
processed_data = {}

for key, filename in files.items():
    try:
        path = os.path.join(RAW_PATH, filename)
        raw_df = pd.read_excel(path)

        clean_df = robust_clean_hcp(raw_df)

        if 'gdp' in key:
            growth_col = find_growth_column(clean_df)
            clean_df['GDP_Index'] = build_index(clean_df[growth_col])
            clean_df = clean_df[['Year', 'GDP_Index']]

        processed_data[key] = clean_df
        print(f"✅ {key}")

    except Exception as e:
        print(f"❌ {key} : {e}")


# --- 5. MASTER MERGE ---

# Total GDP
master_df = processed_data['gdp_total'][['Year', 'GDP_Index']].copy()
master_df = master_df.rename(columns={'GDP_Index': 'Y_Total'})

# Sector GDPs
for s in ['gdp_primary', 'gdp_secondary', 'gdp_tertiary']:
    temp = processed_data[s].copy()

    sector_name = s.split("_")[1].capitalize()
    temp = temp.rename(columns={'GDP_Index': f'Y_{sector_name}'})

    master_df = master_df.merge(temp, on='Year', how='inner')

# Employment
emp_df = processed_data['emp_branches'].copy()
sector_cols = find_sector_columns(emp_df.columns)

emp_df = emp_df[['Year'] + list(sector_cols.values())]
master_df = master_df.merge(emp_df, on='Year', how='inner')


# --- 6. EMPLOYMENT LEVELS ---
TOTAL_EMP_2024 = 10673000

master_df['L_Primary'] = (master_df[sector_cols['Primary']] / 100) * TOTAL_EMP_2024
master_df['L_Secondary'] = (master_df[sector_cols['Secondary']] / 100) * TOTAL_EMP_2024
master_df['L_Tertiary'] = (master_df[sector_cols['Tertiary']] / 100) * TOTAL_EMP_2024


# --- 7. PRODUCTIVITY ---

# Total employment
master_df['L_Total'] = (
    master_df['L_Primary'] +
    master_df['L_Secondary'] +
    master_df['L_Tertiary']
)

# Raw productivity (Y/L)
master_df['Prod_Total'] = master_df['Y_Total'] / master_df['L_Total']
master_df['Prod_Primary'] = master_df['Y_Primary'] / master_df['L_Primary']
master_df['Prod_Secondary'] = master_df['Y_Secondary'] / master_df['L_Secondary']
master_df['Prod_Tertiary'] = master_df['Y_Tertiary'] / master_df['L_Tertiary']


# --- 8. PRODUCTIVITY NORMALIZATION ---

# A. Scale (readable values)
SCALE = 1_000_000

for col in ['Prod_Total', 'Prod_Primary', 'Prod_Secondary', 'Prod_Tertiary']:
    master_df[f'{col}_Scaled'] = master_df[col] * SCALE


# B. Productivity index (base = most recent year)
base_year = master_df['Year'].max()

for col in ['Prod_Total', 'Prod_Primary', 'Prod_Secondary', 'Prod_Tertiary']:
    base_value = master_df.loc[master_df['Year'] == base_year, col].values[0]
    master_df[f'{col}_Index'] = (master_df[col] / base_value) * 100


# C. Relative productivity (sector vs total)
master_df['Rel_Prod_Primary'] = master_df['Prod_Primary'] / master_df['Prod_Total']
master_df['Rel_Prod_Secondary'] = master_df['Prod_Secondary'] / master_df['Prod_Total']
master_df['Rel_Prod_Tertiary'] = master_df['Prod_Tertiary'] / master_df['Prod_Total']


# --- 9. STRUCTURAL TRANSFORMATION ---

master_df['Share_L_Primary'] = master_df['L_Primary'] / master_df['L_Total']
master_df['Share_L_Secondary'] = master_df['L_Secondary'] / master_df['L_Total']
master_df['Share_L_Tertiary'] = master_df['L_Tertiary'] / master_df['L_Total']


# --- 10. CLEAN FINAL DATASET (OPTIONAL BUT RECOMMENDED) ---

final_cols = [
    'Year',
    'Y_Total', 'Y_Primary', 'Y_Secondary', 'Y_Tertiary',
    'L_Total', 'L_Primary', 'L_Secondary', 'L_Tertiary',
    'Prod_Total_Scaled', 'Prod_Primary_Scaled',
    'Prod_Secondary_Scaled', 'Prod_Tertiary_Scaled',
    'Prod_Total_Index', 'Prod_Primary_Index',
    'Prod_Secondary_Index', 'Prod_Tertiary_Index',
    'Rel_Prod_Primary', 'Rel_Prod_Secondary', 'Rel_Prod_Tertiary',
    'Share_L_Primary', 'Share_L_Secondary', 'Share_L_Tertiary'
]

master_df = master_df[final_cols]


# --- 11. EXPORT ---

output_file = os.path.join(PROCESSED_PATH, "morocco_master_econometrics.csv")
master_df.to_csv(output_file, index=False)

print(f"\n📂 FINAL dataset exported: {output_file}")
print(master_df.head())

🚀 Launching Pipeline...
✅ gdp_primary
✅ gdp_secondary
✅ gdp_tertiary
✅ gdp_total
✅ emp_branches

📂 FINAL dataset exported: ../data/processed/morocco_master_econometrics.csv
     Year     Y_Total   Y_Primary  Y_Secondary  Y_Tertiary     L_Total  \
0  2025.0  134.639637  109.470728   127.427636  138.995408  10673000.0   
1  2024.0  128.595642  105.564829   121.591256  133.009960  10662327.0   
2  2023.0  123.887902  110.539088   116.690265  127.160573  10662327.0   
3  2022.0  119.467601  108.584566   115.764151  121.105308  10662327.0   
4  2021.0  117.355207  123.111753   117.646495  113.394483  10662327.0   

   L_Primary  L_Secondary  L_Tertiary  Prod_Total_Scaled  ...  \
0  2721615.0    2646904.0   5304481.0          12.614976  ...   
1  2806999.0    2582866.0   5272462.0          12.060748  ...   
2  2967094.0    2540174.0   5155059.0          11.619218  ...   
3  3127189.0    2476136.0   5059002.0          11.204646  ...   
4  3329976.0    2444117.0   4888234.0          11.006529 